# Buscacompis
Proyecto de Matemáticas Discretas que te recomienda estudiantes con los que formar un grupo de estudio basado un tu perfil.


In [13]:
import pandas as pd


## 1. Cargar el dataset
Leemos el CSV. Cada estudiante tiene texto separado por `;` en `materias`, `hobbies` y `horario_disponible`.
Los convertimos en conjuntos con la función set() de Python que es la estructura matemática que necesitamos.

In [3]:
# Carga del csv con los datos de prueba
df = pd.read_csv("../data/estudiantes.csv", encoding="latin1", skiprows=1)

# Verificación con las primeras filas
df.head()

,id,nombre,carrera,materias,hobbies,horario_disponible
0,1,Ana Torres,Ingeniería de Sistemas,Cálculo I;Programación I;Física I,ajedrez;lectura;videojuegos,Lun-8am;Mar-2pm;Jue-8am
1,2,Carlos Ruiz,Ingeniería de Sistemas,Cálculo I;Programación I;Matemáticas Discretas I,fútbol;videojuegos;música,Lun-8am;Mie-10am;Vie-2pm
2,3,Laura Gómez,Ingeniería de Sistemas,Cálculo I;Física I;Química,lectura;pintura;cine,Mar-2pm;Jue-8am;Vie-4pm
3,4,Diego Martínez,Ingeniería Industrial,Cálculo I;Álgebra Lineal;Química,fútbol;natación;música,Lun-2pm;Mar-8am;Sab-10am
4,5,María Fernández,Ingeniería de Sistemas,Programación I;Matemáticas Discretas I;Inglés I,ajedrez;videojuegos;cocina,Mie-10am;Jue-8am;Vie-2pm


In [12]:
def texto_a_conjunto(texto):    #Esta función convierte las columnas con varios valores en un conjunto con estos valores
    return set(texto.split(";"))



# Creamos 3 columnas nuevas como conjuntos en vez de texto
df["materias_conjunto"] = df["materias"].apply(texto_a_conjunto)
df["hobbies_conjunto"] = df["hobbies"].apply(texto_a_conjunto)
df["horario_conjunto"] = df["horario_disponible"].apply(texto_a_conjunto)

# Revisamos cómo quedó un estudiante de ejemplo
df.loc[0, ["nombre", "materias_conjunto", "hobbies_conjunto", "horario_conjunto"]]

nombre                                          Ana Torres
materias_conjunto    {Programación I, Cálculo I, Física I}
hobbies_conjunto           {lectura, ajedrez, videojuegos}
horario_conjunto               {Mar-2pm, Jue-8am, Lun-8am}
Name: 0, dtype: object

## 2. Similitud de Jaccard
Para dos conjuntos A y B:

$$ Jaccard(A, B) = \frac{|A \cap B|}{|A \cup B|} $$

Este cálculo nos dice cuántos elementos tienen en común, sobre el total de elementos distintos entre los dos. Osea, nos dice qué tanta similitud existe entre dos conjuntos. Da un número entre 0 y 1: un 0 significa que no tienen nada en común y un 1 significa que los conjuntos son idénticos.
Usaremos este cálculo para saber que tántas cosas tienen en común dos estudiantes entre sí y, definiendo un umbral de aceptación, decidir si pueden formar un grupo de trabajo.

In [5]:
def jaccard(conjunto_a, conjunto_b):         # Esta función calcula la similitud de Jaccard entre dos conjuntos.
                                             #Si ambos conjuntos están vacíos, devolvemos 0 para evitar dividir por cero.
    interseccion = conjunto_a & conjunto_b   # elementos en común
    union = conjunto_a | conjunto_b          # elementos distintos entre los dos
    if len(union) == 0:
        return 0.0
    return len(interseccion) / len(union)

# Prueba con dos conjuntos inventados
prueba_a = {"Cálculo I", "Programación I", "Física I"}
prueba_b = {"Cálculo I", "Programación I", "Matemáticas Discretas I"}
print("Jaccard de prueba:", jaccard(prueba_a, prueba_b))

Jaccard de prueba: 0.5


## 3. Ponderación de similitudes
Hacemos una suma ponderada de las similitudes ya que el horario disponible y las materias son más importantes que los hobbies a la hora de asignar un posible compañero de estudio.
Para este caso definimos un peso de 0.5 para la similitud de las materias, 0.3 para la similitud de horarios y 0.2 para la similitud de hobbies, la fórmula utilizada es la siguiente:
$$Suma = (0.5 \cdot Sim\_materias) + (0.3 \cdot Sim\_horarios) + (0.2 \cdot Sim\_hobbies)$$

In [9]:
def suma_ponderada(estudiante_a, estudiante_b, peso_materias=0.5, peso_horario=0.3, peso_hobbies=0.2): 
#Esta función recibe dos estudianteses y devueve
#una suma ponderada de las compatibilidades de los conjuntos entre 0 y 1.
    sim_materias = jaccard(estudiante_a["materias_conjunto"], estudiante_b["materias_conjunto"])
    sim_horario = jaccard(estudiante_a["horario_conjunto"], estudiante_b["horario_conjunto"])
    sim_hobbies = jaccard(estudiante_a["hobbies_conjunto"], estudiante_b["hobbies_conjunto"])
    return (peso_materias * sim_materias) + (peso_horario * sim_horario) + (peso_hobbies * sim_hobbies)

# Probamos con dos estudiantes
estudiante_1 = df.iloc[0]
estudiante_2 = df.iloc[1]

print(f"Compatibilidad entre {estudiante_1['nombre']} y {estudiante_2['nombre']}: {suma_ponderada(estudiante_1, estudiante_2):.3f}")

Compatibilidad entre Ana Torres y Carlos Ruiz: 0.350


## 4. Pruebas
Calculamos la suma ponderada entre varias parejas de estudiantes para verificar que los números tengan sentido y que los 
estudiantes con materias parecidas deberían dar scores más altos.

In [11]:
parejas_prueba = [(0, 1), (0, 5), (7, 10), (0, 19)]

for i, j in parejas_prueba:
    a = df.iloc[i]
    b = df.iloc[j]
    s = suma_ponderada(a, b)
    print(f"{a['nombre']:20s} <-> {b['nombre']:20s}  score = {s:.3f}")

Ana Torres           <-> Carlos Ruiz           score = 0.350
Ana Torres           <-> Andrés López          score = 0.690
Santiago Vargas      <-> Isabella Moreno       score = 0.500
Ana Torres           <-> Felipe Aguilar        score = 0.000
